# Feature Importance

**Topic:** Which Variables Drive Predictions

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** how tree-based importance and permutation importance measure feature relevance differently
- **Explain** why adding noise features can inflate importance scores for signal features
- **Interpret** a feature importance chart and identify which features are genuinely predictive

> **Tip:** Start with 0 noise features and note the signal rankings. Add 10 noise features and watch the signal scores dilute — under tree-based importance the bars must shrink because the scores always sum to 1. Then turn on **Add high-cardinality ID** and compare the two methods on that column: tree-based importance ranks the random ID near the top, while permutation importance correctly ignores it. That gap is the single most important lesson here.

---
## How we got here

In **09_the_no_free_lunch_theorem.ipynb** we established that algorithm choice depends on data structure. But once a model is trained, a new question arises: *which features actually contributed to the predictions?*

Feature importance connects directly to the feature_engineering/ folder coming up — understanding which features matter is the first step toward building better ones. It also connects back to preprocessing/: knowing which features are irrelevant can tell you what to remove.

---
## Why this matters for data science

Feature importance serves three purposes:
1. **Model understanding:** Explain predictions to stakeholders who need to trust the model
2. **Feature selection:** Remove irrelevant features to reduce noise and training time
3. **Debugging:** A model where "customer ID" ranks as the top feature is almost certainly overfitting

Domain knowledge still matters: a model might rank a noise feature highly if it correlates with the target in the training set by chance. Importance scores are evidence, not ground truth.

---
## Try it yourself

In [2]:
np.random.seed(42)
from sklearn.model_selection import train_test_split

out = Output()

noise_slider = IntSlider(value=0, min=0, max=15, step=1,
    description="Noise features:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"))

method_toggle = ToggleButtons(
    options=["Tree-based importance", "Permutation importance"],
    value="Tree-based importance",
    description="", layout=widgets.Layout(width="420px"))

idcol_toggle = ToggleButtons(
    options=["No ID feature", "Add high-cardinality ID"],
    value="No ID feature",
    description="", layout=widgets.Layout(width="420px"))

corr_toggle = ToggleButtons(
    options=["No correlated copy", "Add correlated copy of Signal 1"],
    value="No correlated copy",
    description="", layout=widgets.Layout(width="420px"))

caption = widgets.HTML(layout=widgets.Layout(width="720px"))

BASE_FEATURES = 5
N = 300

# Signal features + labels generated ONCE, so the signal columns are identical
# at every slider position — dilution becomes a true before/after, not a resample.
X_SIG, Y = make_classification(n_samples=N, n_features=BASE_FEATURES,
    n_informative=BASE_FEATURES, n_redundant=0, random_state=42)

def render(n_noise, method, id_mode, corr_mode):
    rng = np.random.RandomState(0)
    X = X_SIG.copy()
    feature_names = [f"Signal {i+1}" for i in range(BASE_FEATURES)]

    add_corr = corr_mode.startswith("Add correlated")
    if add_corr:
        # Near-duplicate of Signal 1: same column + small noise (~0.97 corr).
        copy_col = (X_SIG[:, 0] + rng.normal(0, 0.15, size=N)).reshape(-1, 1)
        X = np.hstack([X, copy_col])
        feature_names += ["Signal 1-copy"]

    if n_noise:
        X = np.hstack([X, rng.normal(size=(N, n_noise))])
        feature_names += [f"Noise {i+1}" for i in range(n_noise)]

    add_id = id_mode == "Add high-cardinality ID"
    if add_id:
        id_col = rng.permutation(N).reshape(-1, 1).astype(float)
        X = np.hstack([X, id_col])
        feature_names += ["ID (unique)"]

    n_total = X.shape[1]
    X_tr, X_te, y_tr, y_te = train_test_split(X, Y, test_size=0.3,
        random_state=42, stratify=Y)
    clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)

    if method == "Tree-based importance":
        importances = clf.feature_importances_
        xaxis_label = "Importance (impurity decrease, sums to 1)"
    else:
        result = permutation_importance(clf, X_te, y_te,
            n_repeats=10, random_state=42)
        importances = result.importances_mean
        xaxis_label = "Importance (accuracy drop when shuffled)"

    order = np.argsort(importances)
    def color_for(nm):
        if nm == "Signal 1-copy": return PALETTE["muted"]
        if nm.startswith("Signal"): return PALETTE["primary"]
        if nm.startswith("ID"):     return PALETTE["accent"]
        return PALETTE["secondary"]
    colors = [color_for(feature_names[i]) for i in order]

    traces = [go.Bar(
        x=importances[order],
        y=[feature_names[i] for i in order],
        orientation="h", marker_color=colors, showlegend=False)]
    extras = f"{' | +corr' if add_corr else ''}{' | +ID' if add_id else ''}"
    layout = base_layout(
        title=f"{method} | {n_noise} noise feature(s){extras}",
        xaxis_title=xaxis_label, yaxis_title="")
    layout.update(height=max(320, 30*n_total + 80))
    if method == "Permutation importance":
        layout.update(xaxis=dict(zeroline=True, zerolinewidth=1.5,
                                 zerolinecolor=PALETTE["muted"]))

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(go.Figure(data=traces, layout=layout)))

    # ---- Dynamic caption ----
    parts = []
    sig_idx = [i for i, nm in enumerate(feature_names) if nm == f"Signal 1" or nm == "Signal 1-copy"]

    if method == "Tree-based importance":
        sig_all = [i for i, nm in enumerate(feature_names) if nm.startswith("Signal") and nm != "Signal 1-copy"]
        parts.append("<b>Tree-based (MDI)</b> scores sum to 1, so adding features shrinks "
                     "every existing bar — that's the dilution.")
    else:
        parts.append("<b>Permutation</b> scores are accuracy drops on held-out data, not "
                     "normalized — compare <i>rankings</i> across methods, never magnitudes. "
                     "Noise features sit near zero (sometimes slightly negative).")

    if add_corr:
        i_orig = feature_names.index("Signal 1")
        i_copy = feature_names.index("Signal 1-copy")
        v_orig, v_copy = importances[i_orig], importances[i_copy]
        if method == "Tree-based importance":
            parts.append(f"<br><br><b>Correlated pair:</b> Signal 1 and its copy are ~97% correlated, "
                         f"so the forest splits the credit between them somewhat arbitrarily "
                         f"({v_orig:.3f} vs {v_copy:.3f}). Each looks weaker than Signal 1 did alone, "
                         "even though the underlying signal is unchanged — the importance was halved, "
                         "not destroyed.")
        else:
            parts.append(f"<br><br><b>Correlated pair:</b> here's permutation's blind spot. Shuffling "
                         f"Signal 1 alone barely dents accuracy ({v_orig:.3f}) because the model just "
                         f"reads the signal off its copy instead — and vice versa ({v_copy:.3f}). Both "
                         "look unimportant, yet together they carry real signal. Permutation "
                         "<i>underestimates</i> correlated features; this is the caveat to remember.")

    if add_id:
        id_i = feature_names.index("ID (unique)")
        id_rank = list(np.argsort(importances)[::-1]).index(id_i) + 1
        if method == "Tree-based importance":
            parts.append(f"<br><br><b>ID feature:</b> pure random noise, yet MDI ranks it "
                         f"#{id_rank} of {n_total} — the high-cardinality trap.")
        else:
            parts.append(f"<br><br><b>ID feature:</b> permutation correctly drops it near zero "
                         f"(#{id_rank} of {n_total}).")

    caption.value = (f"<div style='font-size:13px; line-height:1.5; padding:6px 2px'>"
                     f"{''.join(parts)}</div>")

def on_change(change):
    render(noise_slider.value, method_toggle.value,
           idcol_toggle.value, corr_toggle.value)
for w in (noise_slider, method_toggle, idcol_toggle, corr_toggle):
    w.observe(on_change, names="value")
display(VBox([noise_slider, method_toggle, idcol_toggle, corr_toggle, out, caption]))
render(0, "Tree-based importance", "No ID feature", "No correlated copy")

---
## What's happening?

Diagnosing a car problem: some symptoms are highly informative (engine light, unusual sound), others are irrelevant (seat color, air freshener scent). Feature importance asks the same question of a model: which inputs actually influenced the output?

**Tree-based importance** (mean decrease in impurity) measures how much each feature reduced uncertainty across all the tree's splits. It is fast and built into tree models, but it has two quirks the widget makes visible: the scores always sum to 1 (so adding *any* feature shrinks the others — that's the dilution you see when you add noise), and it inflates **high-cardinality** features that offer many split points.

**Permutation importance** shuffles one feature at a time and measures how much the model's accuracy drops on held-out data. If shuffling a feature makes no difference, it was not contributing. This method is model-agnostic and more reliable — and because it's measured by accuracy drop rather than impurity, its scores are on a completely different scale.

**The two experiments to run:**

- **Dilution (noise slider):** the five signal features are held fixed across every slider position, so as you add noise you're watching the *same* features lose share. Under tree-based importance they shrink because the total is capped at 1; the ranking usually survives, but the magnitudes compress.
- **The ID trap (ID toggle):** the "ID (unique)" column is pure random noise with a unique value per row. Tree-based importance ranks it deceptively high — endless split points let it cut the training data finely. Permutation importance drops it to ~zero, because shuffling random IDs costs no real accuracy. Toggling between the two methods on this one column is the cleanest demonstration of why you don't trust impurity-based importance alone.

> **Reading the scores:** rankings are comparable across the two methods; **magnitudes are not.** A signal feature might read 0.18 under tree-based importance and 0.04 under permutation — that's a change of units, not a disagreement about importance. And under permutation, noise features hover around zero and occasionally go *slightly negative* — meaning shuffling them happened to *help*, the tell-tale fingerprint of a useless feature.

| Method | How it works | Model agnostic? | Risk | sklearn implementation |
|---|---|---|---|---|
| Tree-based (MDI) | Reduction in impurity across splits | No (trees only) | Inflates high-cardinality features; scores sum to 1 so they dilute | `clf.feature_importances_` |
| Permutation | Accuracy drop when feature is shuffled on held-out data | Yes | Underestimates correlated features — shuffling one leaves its twin to carry the signal, so both look unimportant | `permutation_importance()` |
| SHAP values | Shapley game-theory allocation | Yes | Computationally expensive | `shap` library |

---
## Real-world example: Loan approval model

A bank's gradient boosting model assigns high importance to "zip code" and "application timestamp." Zip code is a proxy for race — using it may be illegal and will generalize poorly. Application timestamp is leaking information (loans applied for at 3 AM have very different approval rates).

Feature importance surfaced both problems before deployment. Without this analysis, the model would have shipped with discriminatory features and a data leak.

---
## Key takeaway

> **Feature importance tells you what the model actually used — and a high importance score for a suspicious feature is a debugging signal, not a reason to celebrate.**

---
*Next up: The Curse of Dimensionality — why having more features can actually make your model worse*